### Dreiecke zählen – Lösung Schritt für Schritt

*36. Bundeswettbewerb Informatik (2017), 1. Runde, Aufgabe 3*

Janina hat von ihrer Freundin Nadine ein Rätsel aus dem Internet zugeschickt bekommen: Wie viele Dreiecke sind in einer Zeichnung aus Strecken zu sehen? Sie will sich nicht jedes Mal ans Zählen machen und überlegt, ob sich solche Rätsel mit einem Computer lösen lassen.

#### Aufgabe 3

Schreibe ein Programm, das die Dreiecke in einer Rätsel-Zeichnung zählt. Eine Zeichnung besteht aus einigen Strecken. Du kannst davon ausgehen, dass keine zwei Strecken auf derselben Geraden liegen und dass sich nie mehr als zwei Strecken im gleichen Punkt schneiden. In der Beispielzeichnung der Aufgabe (`dreiecke1.txt`, 6 Strecken) sind neun Dreiecke zu finden.

Wende dein Programm auf die Beispiele an, die du auf den BwInf-Webseiten findest.

#### Eingabeformat (`beispieldaten/dreiecke1.txt` bis `dreiecke6.txt`)

Erste Zeile: Anzahl der Strecken. Danach je Zeile eine Strecke `x1 y1 x2 y2` (Koordinaten der beiden Endpunkte als Kommazahlen).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zerlegen in zwei Schritte:** Erst für jedes Streckenpaar feststellen, ob und wo es sich schneidet. Dann aus diesen Ergebnissen die Dreiecke zählen.
- **Dreieck = drei Strecken, die sich paarweise schneiden.** Drei Strecken bilden ein Dreieck, wenn sich jedes Paar von ihnen trifft und die drei Treffpunkte **verschieden** sind. Sind sie gleich, laufen die drei Strecken nur durch denselben Punkt (wie die Strahlen eines Sterns).
- **Schnittpunkt zweier Strecken:** Eine Strecke beschreibt man als Startpunkt plus Vielfaches einer Richtung. Zwei Strecken schneiden sich, wenn beide Vielfache zwischen 0 und 1 liegen. Die Rechnung nutzt das Kreuzprodukt zweier Vektoren.
- **Kommazahlen vergleicht man nicht mit `==`**, sondern mit einer kleinen Toleranz.
- **Alle Tripel durchgehen** mit drei geschachtelten Schleifen (`i < j < k`), damit jedes Dreieck nur einmal gezählt wird.

**Nützliche Funktionen und Bibliotheken**
- Eine Matrix als Liste von Listen für die Schnittpunkte.
- `abs` für den Vergleich mit Toleranz.
- Optional `itertools.combinations(range(n), 3)` statt der drei Schleifen.

## Die Idee

1. **Schnittpunkte:** Für jedes Paar von Strecken `(i, j)` berechnen wir, ob sie sich schneiden und wo. Das Ergebnis speichern wir in einer Matrix `S[i][j]` (der Schnittpunkt oder `None`).
2. **Dreiecke:** Für jedes Tripel `i < j < k` prüfen wir, ob alle drei Paare einen Schnittpunkt haben (`S[i][j]`, `S[j][k]`, `S[i][k]`) und ob diese drei Punkte **paarweise verschieden** sind.

Die Aufgabe garantiert, dass sich nie mehr als zwei Strecken in einem berechneten Schnittpunkt treffen. Die Beispieldaten 5 und 6 enthalten aber Strecken, die einen **gemeinsamen Endpunkt** haben (z. B. drei Strecken, die im selben Punkt starten). Dann schneiden sich alle drei Paare, aber im selben Punkt: kein Dreieck, nur ein „Dreieck mit Fläche 0“. Deshalb ist die Prüfung auf verschiedene Punkte nötig.

## Schritt 1: Daten einlesen

Eine Strecke speichern wir als Tupel `(x1, y1, x2, y2)`.

In [ ]:
def lies_strecken(dateiname):
    """Gibt die Strecken der Datei als Liste von Tupeln (x1, y1, x2, y2) zurück."""
    with open(dateiname) as datei:
        zeilen = datei.read().split('\n')
    anzahl = int(zeilen[0])
    strecken = []
    for zeile in zeilen[1:anzahl + 1]:
        strecken.append(tuple(float(z) for z in zeile.split()))
    return strecken


strecken = lies_strecken('beispieldaten/dreiecke1.txt')
print(len(strecken), 'Strecken:')
for s in strecken:
    print(s)

6 Strecken:
(0.0, 0.0, 0.0, 200.0)
(0.0, 0.0, 120.0, 0.0)
(120.0, 0.0, 0.0, 200.0)
(0.0, 20.0, 100.0, 180.0)
(20.0, 0.0, 50.0, 180.0)
(100.0, 0.0, 50.0, 180.0)


## Schritt 2: Schnittpunkt zweier Strecken

Eine Strecke mit den Endpunkten `a` und `b` schreiben wir als **Startpunkt** `p = a` und **Richtung** `r = b - a`. Der Punkt `p + t·r` liegt für `t` zwischen 0 und 1 auf der Strecke (`t = 0` ist `a`, `t = 1` ist `b`). Eine zweite Strecke mit Startpunkt `q` und Richtung `s` liefert genauso `q + u·s`.

Am Schnittpunkt gilt `p + t·r = q + u·s`. Mit dem Kreuzprodukt zweier Vektoren, `v × w = v_x·w_y - v_y·w_x`, kann man diese Gleichung nach `t` und `u` auflösen (die Herleitung steht in den Lösungshinweisen):

- `t = ((q - p) × s) / (r × s)`
- `u = ((q - p) × r) / (r × s)`

Der Nenner `r × s` ist 0, wenn die Strecken parallel sind. Dann gibt es keinen Schnittpunkt (liegen sie auf derselben Geraden, überlappen sie in den Beispielen nicht). Sonst schneiden sich die **Strecken** genau dann, wenn `t` **und** `u` zwischen 0 und 1 liegen. Wegen der Rundung von Kommazahlen lassen wir einen winzigen Spielraum `EPS` zu, damit sich zwei Strecken mit gemeinsamem Endpunkt sicher schneiden.

In [ ]:
EPS = 1e-9


def schnittpunkt(a, b):
    """Gibt den Schnittpunkt (x, y) der Strecken a und b zurück, oder None, wenn sie sich nicht schneiden."""
    px, py = a[0], a[1]                       # Startpunkt p der Strecke a
    rx, ry = a[2] - a[0], a[3] - a[1]         # Richtung r
    qx, qy = b[0], b[1]                       # Startpunkt q der Strecke b
    sx, sy = b[2] - b[0], b[3] - b[1]         # Richtung s
    nenner = rx * sy - ry * sx                # r x s
    if abs(nenner) < 1e-12:                   # parallel: kein Schnittpunkt
        return None
    dx, dy = qx - px, qy - py                 # q - p
    t = (dx * sy - dy * sx) / nenner          # (q - p) x s / (r x s)
    u = (dx * ry - dy * rx) / nenner          # (q - p) x r / (r x s)
    if -EPS <= t <= 1 + EPS and -EPS <= u <= 1 + EPS:
        return (px + t * rx, py + t * ry)
    return None


# kleine Tests
print(schnittpunkt((0, 0, 10, 10), (0, 10, 10, 0)))        # Kreuz: (5, 5)
print(schnittpunkt((0, 0, 10, 0), (0, 5, 10, 5)))          # parallel: None
print(schnittpunkt((0, 0, 4, 4), (10, 0, 6, 4)))           # Geraden schneiden sich, Strecken nicht: None
print(schnittpunkt((0, 0, 4, 4), (4, 4, 8, 0)))            # gemeinsamer Endpunkt: (4, 4)

(5.0, 5.0)
None
None
(4.0, 4.0)


## Schritt 3: Alle Schnittpunkte

Wir gehen alle Paare `i < j` durch und tragen den Schnittpunkt **symmetrisch** in die Matrix ein: `S[i][j]` und `S[j][i]` sind gleich. Bei `n` Strecken sind das `n·(n-1)/2` Berechnungen.

In [ ]:
def alle_schnittpunkte(strecken):
    """Gibt die Matrix S zurück: S[i][j] ist der Schnittpunkt der Strecken i und j oder None."""
    n = len(strecken)
    S = [[None] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            S[i][j] = schnittpunkt(strecken[i], strecken[j])
            S[j][i] = S[i][j]
    return S


S = alle_schnittpunkte(strecken)
paare = sum(1 for i in range(len(S)) for j in range(i + 1, len(S)) if S[i][j] is not None)
print(len(strecken), 'Strecken,', paare, 'von', len(strecken) * (len(strecken) - 1) // 2, 'Paaren schneiden sich')
for i in range(len(S)):
    print(i, ['--' if p is None else '(%.1f, %.1f)' % p for p in S[i]])

6 Strecken, 12 von 15 Paaren schneiden sich
0 ['--', '(0.0, 0.0)', '(0.0, 200.0)', '(0.0, 20.0)', '--', '--']
1 ['(0.0, 0.0)', '--', '(120.0, 0.0)', '--', '(20.0, 0.0)', '(100.0, 0.0)']
2 ['(0.0, 200.0)', '(120.0, 0.0)', '--', '(55.1, 108.2)', '(41.7, 130.4)', '(82.8, 62.1)']
3 ['(0.0, 20.0)', '--', '(55.1, 108.2)', '--', '(31.8, 70.9)', '(65.4, 124.6)']
4 ['--', '(20.0, 0.0)', '(41.7, 130.4)', '(31.8, 70.9)', '--', '(50.0, 180.0)']
5 ['--', '(100.0, 0.0)', '(82.8, 62.1)', '(65.4, 124.6)', '(50.0, 180.0)', '--']


## Schritt 4: Dreiecke zählen

Für jedes Tripel `i < j < k` prüfen wir:

1. Alle drei Paare schneiden sich (`S[i][j]`, `S[j][k]`, `S[i][k]` sind nicht `None`).
2. Die drei Schnittpunkte sind **paarweise verschieden**. Wegen der Rundung heißt „gleich“: x und y unterscheiden sich um weniger als `0.000001`.

Die Funktion hat einen Schalter `mit_flaeche_null`. Er zählt zusätzlich die Fälle, in denen alle drei Punkte zusammenfallen. Das brauchen wir nur, um die Zahlen der Lösungshinweise nachzuvollziehen.

In [ ]:
def gleich(p, q):
    """True, wenn die Punkte p und q (bis auf Rundung) übereinstimmen."""
    return abs(p[0] - q[0]) < 1e-6 and abs(p[1] - q[1]) < 1e-6


def zaehle_dreiecke(S, mit_flaeche_null=False):
    """Zählt die Tripel i < j < k, deren drei Paare sich schneiden (und deren Schnittpunkte verschieden sind)."""
    n = len(S)
    anzahl = 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                a, b, c = S[i][j], S[j][k], S[i][k]
                if a is None or b is None or c is None:
                    continue
                if mit_flaeche_null or not (gleich(a, b) or gleich(b, c) or gleich(a, c)):
                    anzahl = anzahl + 1
    return anzahl


print('dreiecke1:', zaehle_dreiecke(S))

dreiecke1: 9


## Schritt 5: Alle Beispiele

Wir fassen die Schritte in einer Funktion zusammen und wenden sie auf alle sechs Beispiele an. Die letzte Spalte zählt zusätzlich die Fälle mit Fläche 0 (drei Strecken durch einen Punkt).

In [ ]:
def dreiecke_in_datei(dateiname, mit_flaeche_null=False):
    """Liest die Datei und gibt die Zahl der Dreiecke zurück."""
    return zaehle_dreiecke(alle_schnittpunkte(lies_strecken(dateiname)), mit_flaeche_null)


print('Beispiel  Strecken  Dreiecke  (mit Fläche 0)')
for nummer in range(1, 7):
    name = 'beispieldaten/dreiecke' + str(nummer) + '.txt'
    print('%8d  %8d  %8d  %14d' % (nummer, len(lies_strecken(name)), dreiecke_in_datei(name),
                                   dreiecke_in_datei(name, True)))

Beispiel  Strecken  Dreiecke  (mit Fläche 0)
       1         6         9               9
       2         7         0               0
       3        10         3               3
       4        13         5               5
       5        18         1               4
       6        20        20              25


Die Lösungshinweise nennen **9, 0, 3, 5, 1 und 20** Dreiecke, und wenn man Dreiecke mit Fläche 0 mitzählt, 9, 0, 3, 5, 4 und 25. Das stimmt mit unseren Zahlen überein. Bei den Beispielen 5 und 6 sieht man also, wie wichtig die Prüfung auf verschiedene Punkte ist: Ohne sie wären es 4 statt 1 und 25 statt 20 Dreiecke, weil dort mehrere Strecken denselben Endpunkt haben. Beispiel 2 hat gar keine Dreiecke, weil nie drei Strecken paarweise zusammenstoßen.

Für Beispiel 1 gibt es auch die Lösungsdatei `dreiecke1-loes.txt` mit allen neun Dreiecken (je Zeile die drei Eckpunkte). Wir prüfen, ob unser Programm genau diese Dreiecke findet. Dazu bestimmen wir die Eckpunkte jedes gefundenen Dreiecks (auf zwei Stellen gerundet) und vergleichen sie mit der Datei.

In [ ]:
def dreieck_ecken(S):
    """Gibt für jedes Dreieck die sortierte Liste seiner drei Eckpunkte (auf zwei Stellen gerundet) zurück."""
    n = len(S)
    ergebnis = []
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                a, b, c = S[i][j], S[j][k], S[i][k]
                if a is None or b is None or c is None:
                    continue
                if not (gleich(a, b) or gleich(b, c) or gleich(a, c)):
                    ecken = sorted((round(p[0], 2) + 0.0, round(p[1], 2) + 0.0) for p in (a, b, c))
                    ergebnis.append(ecken)
    return sorted(ergebnis)


def lies_loesung(dateiname):
    """Liest die Lösungsdatei: erste Zeile Anzahl, dann je Zeile 3 Eckpunkte (x1 y1 x2 y2 x3 y3)."""
    with open(dateiname) as datei:
        zeilen = datei.read().split('\n')
    ergebnis = []
    for zeile in zeilen[1:int(zeilen[0]) + 1]:
        z = [float(w) for w in zeile.split()]
        ergebnis.append(sorted([(z[0], z[1]), (z[2], z[3]), (z[4], z[5])]))
    return sorted(ergebnis)


unsere = dreieck_ecken(S)
offiziell = lies_loesung('beispieldaten/dreiecke1-loes.txt')
passt = 0
for d in unsere:
    for e in offiziell:
        # jede Ecke von d muss (fast) bei einer Ecke von e liegen
        if all(any(abs(p[0] - q[0]) < 0.3 and abs(p[1] - q[1]) < 0.3 for q in e) for p in d):
            passt = passt + 1
print(len(unsere), 'gefundene Dreiecke,', len(offiziell), 'in der Lösungsdatei,', passt, 'stimmen überein')
print('Das erste Dreieck:', unsere[0])

9 gefundene Dreiecke, 9 in der Lösungsdatei, 9 stimmen überein
Das erste Dreieck: [(0.0, 0.0), (0.0, 200.0), (120.0, 0.0)]


Alle neun Dreiecke stimmen mit der Lösungsdatei überein. Wir haben dabei eine Toleranz von `0.3` erlaubt, denn die Lösungsdatei nennt einmal den Punkt `(0, 20.28)`, wo die Strecke `0 20 100 180` die Strecke an der y-Achse trifft. Nachrechnen ergibt genau `(0, 20)`, das ist ein Fehler in der Lösungsdatei. Unser Programm rechnet hier richtig.

## Zum Rechenaufwand

Unser Verfahren prüft alle Tripel. Bei `n` Strecken sind das `n·(n-1)·(n-2)/6`, also etwa `n³/6` Tripel (die Laufzeit wächst „kubisch“). Bei den Beispielen mit höchstens 20 Strecken sind es höchstens 1140 Tripel, das ist kein Problem. Hätte eine Zeichnung 1000 Strecken, wären es schon rund 166 Millionen Tripel, davon die allermeisten ohne Dreieck.

Auch die Schnittpunkte aller Paare (`n²/2` Berechnungen) sind für die Beispiele völlig ausreichend.

## Für Neugierige: nur die Nachbarn betrachten

Die meisten Tripel scheitern schon beim ersten Paar. Wir können das ausnutzen: Wir merken uns für jede Strecke die **Liste ihrer Nachbarn**, also der Strecken, die sie schneiden. Ein Dreieck `i < j < k` entsteht dann so: `j` ist ein Nachbar von `i` (mit `j > i`), `k` ist ein Nachbar von `j` (mit `k > j`), und zuletzt prüfen wir nur noch, ob auch `i` und `k` sich schneiden. Das spart alle Tripel, deren erstes oder zweites Paar sich nicht schneidet. Zur Kontrolle zählen wir, wie viele Tripel jeweils angeschaut werden.

In [ ]:
def zaehle_dreiecke_nachbarn(S):
    """Zählt die Dreiecke wie zaehle_dreiecke, schaut aber nur Tripel an, deren erste beide Paare sich schneiden.
    Gibt (Anzahl der Dreiecke, Anzahl der angeschauten Tripel) zurück."""
    n = len(S)
    nachbarn = []
    for i in range(n):
        nachbarn.append([j for j in range(n) if S[i][j] is not None])
    anzahl = 0
    angeschaut = 0
    for i in range(n):
        for j in nachbarn[i]:
            if j <= i:
                continue
            for k in nachbarn[j]:
                if k <= j:
                    continue
                angeschaut = angeschaut + 1
                if S[i][k] is not None:
                    a, b, c = S[i][j], S[j][k], S[i][k]
                    if not (gleich(a, b) or gleich(b, c) or gleich(a, c)):
                        anzahl = anzahl + 1
    return anzahl, angeschaut


print('Beispiel  Dreiecke  angeschaute Tripel  alle Tripel')
for nummer in range(1, 7):
    ss = alle_schnittpunkte(lies_strecken('beispieldaten/dreiecke' + str(nummer) + '.txt'))
    n = len(ss)
    dreiecke, angeschaut = zaehle_dreiecke_nachbarn(ss)
    assert dreiecke == zaehle_dreiecke(ss)
    print('%8d  %8d  %18d  %11d' % (nummer, dreiecke, angeschaut, n * (n - 1) * (n - 2) // 6))

Beispiel  Dreiecke  angeschaute Tripel  alle Tripel
       1         9                  16           20
       2         0                   1           35
       3         3                   7          120
       4         5                  13          286
       5         1                  21          816
       6        20                  69         1140


Dieselben Dreiecke, aber mit weniger angeschauten Tripeln. Noch schneller geht es mit zwei Ideen aus den Lösungshinweisen:

- **Schnittpunkte schneller finden:** Mit dem Bentley-Ottmann-Algorithmus („Sweepline“: eine senkrechte Linie wandert von links nach rechts über die Zeichnung) findet man alle Schnittpunkte in `O(n·log n + k)` statt `O(n²)`, wenn es `k` Schnittpunkte gibt. Der Gedanke dahinter: Zwei Strecken können sich nur schneiden, wenn sie vorher an der wandernden Linie direkt nebeneinander lagen.
- **Dreiecke im Schnittpunkt-Graphen schneller finden:** Betrachtet man Strecken als Knoten und Schnittpunkte als Kanten, sucht man Dreiecke in einem Graphen. Dafür gibt es Verfahren (Chiba und Nishizeki), die Knoten mit hohem Grad zuerst behandeln und entfernen.

Für die Größe der Wettbewerbsbeispiele lohnt sich das nicht.

## Zum Schluss: MiniZinc?

Wir haben geprüft, ob sich die Aufgabe sinnvoll mit MiniZinc lösen lässt, und es bei **allen sechs Beispielen** ausprobiert: Ein Modell, das aus der Schnittmatrix die Tripel zählt (eine Summe über alle `i < j < k`), liefert 9, 0, 3, 5, 1 und 20 in jeweils unter 0,2 Sekunden. Es ist aber nur eine **Summe**, bei der der Solver nichts suchen oder optimieren muss, und die eigentliche Arbeit, die Schnittpunkte zu berechnen, müssten wir doch in Python erledigen und als Datendatei übergeben. Ein Constraint-Solver bringt hier keinen Gewinn, deshalb gibt es in diesem Notebook keinen MiniZinc-Abschnitt.